# Archived research notebook

Original filename: `preprocessing sixth低通滤波和下采样（非必要）.ipynb`  
Purpose: Optional low-pass filtering and downsampling.  
Historical research version retained for traceability. The original test data are unavailable; legacy outputs, execution state, and attachments were removed for public release.


In [ ]:
import mne
import os
import numpy as np
from pykalman import KalmanFilter
from mne.preprocessing import ICA
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from scipy import fftpack  # 导入用于Stockwell变换

# 文件夹路径和参数设置
folder_path = r"data/chb-mit/chb01"
target_sfreq = 128  # 目标采样率
lowpass_freq = 80  # 低通滤波截止频率
epoch_duration = 60.0  # 默认窗口长度（秒）
pre_seizure_time = 60.0  # 发作前时间（秒）
post_seizure_time = 60.0  # 发作后时间（秒）

def apply_kalman_filter(data, R=0.01):
    """卡尔曼滤波基线校正"""
    kf = KalmanFilter(
        initial_state_mean=0,
        initial_state_covariance=1,
        transition_matrices=1,
        observation_matrices=1,
        observation_covariance=R,
        transition_covariance=0.001
    )
    state_means, _ = kf.filter(data)
    return data - state_means.flatten()

def apply_zscore(raw_obj):
    """对连续数据（Raw对象）进行Z-分数归一化（按通道）"""
    data = raw_obj.get_data()
    mean = np.mean(data, axis=1, keepdims=True)  # 按通道计算均值
    std = np.std(data, axis=1, keepdims=True)     # 按通道计算标准差
    std[std == 0] = 1.0
    normalized_data = (data - mean) / std
    raw_obj._data = normalized_data
    return raw_obj

def get_seizure_annotations(raw):
    """提取癫痫发作的注释，返回所有发作的开始和结束时间"""
    seizures = []
    if hasattr(raw, 'annotations') and len(raw.annotations) > 0:
        for annot in raw.annotations:
            description = annot['description'].lower() if isinstance(annot['description'], str) else str(annot['description']).lower()
            # 查找可能的癫痫发作注释关键词
            if any(keyword in description for keyword in ['seizure', 'seiz', 'ictal', '发作', '癫痫']):
                onset = annot['onset']
                duration = annot['duration']
                seizures.append((onset, onset + duration))
                print(f"发现癫痫注释: {description}, 开始时间: {onset}秒, 持续时间: {duration}秒")
    return seizures

def create_epochs_with_seizure_info(raw, regular_duration, seizures, pre_time, post_time):
    """创建考虑癫痫发作的时间窗口分段
    
    参数:
        raw: 原始数据对象
        regular_duration: 常规窗口长度（无发作时使用）
        seizures: 癫痫发作时间点列表，每项为(开始时间,结束时间)
        pre_time: 发作前时间（秒）
        post_time: 发作后时间（秒）
    """
    sfreq = raw.info['sfreq']
    data_duration = raw.times[-1]
    
    # 初始化事件列表和标签
    events = []
    event_labels = []  # 0=常规, 1=癫痫前, 2=癫痫发作中, 3=癫痫后
    
    # 如果没有发作标记，或者用户选择忽略标记，则使用常规分段
    if not seizures:
        print("没有发现癫痫发作注释，使用常规分段")
        return create_nonoverlapping_epochs(raw, regular_duration), None
    
    # 当有癫痫发作注释时，创建特殊的epochs
    # 1. 标记所有的发作前期、发作中和发作后期的区间
    marked_regions = []
    for seizure_start, seizure_end in seizures:
        # 确保不超出数据范围
        pre_start = max(0, seizure_start - pre_time)
        post_end = min(data_duration, seizure_end + post_time)
        
        # 添加到标记区域
        marked_regions.append((pre_start, seizure_start, "pre-seizure"))  # 发作前
        marked_regions.append((seizure_start, seizure_end, "seizure"))   # 发作中
        marked_regions.append((seizure_end, post_end, "post-seizure"))   # 发作后
    
    # 2. 对于发作相关区域，创建特定的epoch
    for start, end, label in marked_regions:
        if end > start:  # 确保区间有效
            # 转换为样本点
            start_sample = int(start * sfreq)
            
            # 创建事件
            event_type = {"pre-seizure": 1, "seizure": 2, "post-seizure": 3}[label]
            events.append([start_sample, 0, event_type])
            event_labels.append(label)
            
            print(f"添加 {label} 事件: {start}-{end}秒 (标签={event_type})")
    
    # 3. 对于未被标记的区域，使用常规分段
    # 首先找出所有已标记区域
    all_marked = []
    for start, end, _ in marked_regions:
        all_marked.append((start, end))
    
    # 合并重叠区间
    all_marked.sort()
    merged = []
    for interval in all_marked:
        if not merged or merged[-1][1] < interval[0]:
            merged.append(interval)
        else:
            merged[-1] = (merged[-1][0], max(merged[-1][1], interval[1]))
    
    # 找出未标记区域
    unmarked = []
    last_end = 0
    for start, end in merged:
        if start > last_end:
            unmarked.append((last_end, start))
        last_end = end
    if last_end < data_duration:
        unmarked.append((last_end, data_duration))
    
    # 对未标记区域使用常规分段
    for start, end in unmarked:
        if end - start >= regular_duration:  # 只有当区间足够长时才分段
            # 计算该区间内可以创建的常规epochs数量
            n_epochs = int((end - start) // regular_duration)
            for i in range(n_epochs):
                epoch_start = start + i * regular_duration
                start_sample = int(epoch_start * sfreq)
                events.append([start_sample, 0, 0])  # 标签0表示常规区域
                event_labels.append("regular")
                
                print(f"添加常规事件: {epoch_start}-{epoch_start+regular_duration}秒 (标签=0)")
    
    # 确保events是numpy数组
    events = np.array(events, dtype=int)
    
    # 创建epochs
    if len(events) > 0:
        # 对事件按时间排序
        sort_idx = np.argsort(events[:, 0])
        events = events[sort_idx]
        event_labels = [event_labels[i] for i in sort_idx]
        
        # 创建epochs字典，指定不同事件类型的时间区间
        event_dict = {'regular': 0, 'pre-seizure': 1, 'seizure': 2, 'post-seizure': 3}
        
        # 分别创建和保存不同类型的epochs，不尝试合并它们
        epochs_dict = {}
        
        # 创建常规事件epochs
        regular_events = events[events[:, 2] == 0]
        if len(regular_events) > 0:
            regular_epochs = mne.Epochs(
                raw,
                regular_events,
                event_id={'regular': 0},
                tmin=0,
                tmax=regular_duration,
                baseline=None,
                preload=True,
                verbose=False
            )
            epochs_dict['regular'] = regular_epochs
            print(f"创建了 {len(regular_epochs)} 个常规epochs，每个时长 {regular_duration} 秒")
        
        # 创建发作前epochs
        pre_events = events[events[:, 2] == 1]
        if len(pre_events) > 0:
            pre_epochs = mne.Epochs(
                raw,
                pre_events,
                event_id={'pre-seizure': 1},
                tmin=0,
                tmax=pre_time,
                baseline=None,
                preload=True,
                verbose=False
            )
            epochs_dict['pre-seizure'] = pre_epochs
            print(f"创建了 {len(pre_epochs)} 个发作前epochs，每个时长 {pre_time} 秒")
        
        # 对于发作事件，为每个发作创建单独的epoch
        seizure_epochs_list = []
        for i, (start, end) in enumerate(seizures):
            seizure_duration = end - start
            seizure_event = np.array([[int(start * sfreq), 0, 2]])  # 手动创建事件
            seizure_epoch = mne.Epochs(
                raw,
                seizure_event,
                event_id={'seizure': 2},
                tmin=0,
                tmax=seizure_duration,
                baseline=None,
                preload=True,
                verbose=False
            )
            if len(seizure_epoch) > 0:
                seizure_epochs_list.append(seizure_epoch)
                print(f"创建了发作epoch: {start}-{end}秒，持续时间: {seizure_duration}秒")
        
        if seizure_epochs_list:
            epochs_dict['seizure'] = seizure_epochs_list
        
        # 创建发作后epochs
        post_events = events[events[:, 2] == 3]
        if len(post_events) > 0:
            post_epochs = mne.Epochs(
                raw,
                post_events,
                event_id={'post-seizure': 3},
                tmin=0,
                tmax=post_time,
                baseline=None,
                preload=True,
                verbose=False
            )
            epochs_dict['post-seizure'] = post_epochs
            print(f"创建了 {len(post_epochs)} 个发作后epochs，每个时长 {post_time} 秒")
        
        # 使用常规epochs作为主要对象返回，加上类型字典
        if 'regular' in epochs_dict:
            return epochs_dict['regular'], epochs_dict
        elif len(epochs_dict) > 0:
            # 返回找到的第一种类型
            key = list(epochs_dict.keys())[0]
            if key == 'seizure':
                return seizure_epochs_list[0], epochs_dict
            return epochs_dict[key], epochs_dict
        else:
            print("未能创建有效的epochs，退回到常规分段方法")
            return create_nonoverlapping_epochs(raw, regular_duration), None
    else:
        print("未检测到有效事件，退回到常规分段方法")
        return create_nonoverlapping_epochs(raw, regular_duration), None

def create_nonoverlapping_epochs(raw, duration):
    """创建不重叠的时间窗口分段（用于无癫痫注释的情况）"""
    # 计算可以创建的epochs数量（向下取整）
    data_duration = raw.times[-1]
    n_epochs = int(data_duration // duration)
    
    # 创建人工事件，每个事件间隔duration秒
    events = np.zeros((n_epochs, 3), dtype=int)
    for i in range(n_epochs):
        # 事件样本点位置、前一个事件的ID（0）、当前事件的ID（1）
        events[i, 0] = int(i * duration * raw.info['sfreq'])
        events[i, 2] = 0  # 使用0表示常规区域
    
    # 创建epochs
    epochs = mne.Epochs(
        raw, 
        events, 
        event_id={'regular': 0},
        tmin=0, 
        tmax=duration, 
        baseline=None, 
        preload=True,
        reject=None,
        verbose=False
    )
    return epochs

# 辅助函数：计算ICA成分的峰峰值
def compute_peak_to_peak(component):
    return np.max(component) - np.min(component)

# 辅助函数：计算ICA成分的功率谱密度
def compute_psd(component, sfreq):
    n = len(component)
    fft_result = np.fft.fft(component)
    psd = np.abs(fft_result) ** 2 / n
    freqs = np.fft.fftfreq(n, 1 / sfreq)
    return freqs[:n // 2], psd[:n // 2]

# 辅助函数：判断心电伪迹
def is_ecg_like(psd, freqs):
    powerline_ratio = np.sum(psd[(freqs >= 45) & (freqs <= 65)]) / np.sum(psd)
    return powerline_ratio > 0.1  # 可根据实际情况调整阈值

# Stockwell变换函数
def stockwell_transform(data, fs, fmin=0, fmax=80, n_freq=40):
    """对信号进行Stockwell变换
    
    参数:
        data: 输入信号，形状为(channels, samples)
        fs: 采样率
        fmin: 最小频率
        fmax: 最大频率
        n_freq: 频率点数量
        
    返回:
        st_data: Stockwell变换后的数据，形状为(channels, frequencies, samples)
    """
    n_channels, n_samples = data.shape
    
    # 生成频率点 (对数均匀分布)
    freq_bins = np.logspace(np.log10(max(0.5, fmin)), np.log10(fmax), n_freq)
    
    def st_channel(x):
        # 单通道的Stockwell变换实现
        X = fftpack.fft(x)  # 计算FFT
        n = len(x)
        
        # 预计算所有频率点的指数项
        k = np.arange(n)
        factor = 2 * np.pi * k / n
        
        # 初始化结果数组
        st_channel_data = np.zeros((n_freq, n), dtype=complex)
        
        # 使用贝利分解定理，快速计算变换
        for i, freq in enumerate(freq_bins):
            # 计算标准差 - 高斯窗函数宽度随频率自适应
            sigma = freq / fs
            # 构建高斯窗
            gaussian_window = np.exp(-2 * (np.pi * sigma * k / n) ** 2)
            
            # 使用矩阵操作一次性完成所有时间点的计算
            Xw = X * gaussian_window
            st_channel_data[i, :] = fftpack.ifft(Xw)
            
        return st_channel_data
    
    # 并行处理每个通道
    st_results = Parallel(n_jobs=-1, prefer="threads")(delayed(st_channel)(data[i, :]) for i in range(n_channels))
    st_data = np.array(st_results)
    
    return st_data

# 对epoch应用Stockwell变换的函数
def apply_stockwell_to_epoch(epoch_data, sfreq):
    """对一个epoch应用Stockwell变换"""
    # 获取epoch数据
    data = epoch_data.get_data()  # (epochs, channels, samples)
    n_epochs, n_channels, n_samples = data.shape
    
    # 初始化结果数组
    st_results = []
    
    # 对每个epoch进行Stockwell变换
    for i in range(n_epochs):
        # 应用Stockwell变换
        st_data = stockwell_transform(data[i], fs=sfreq, fmin=0, fmax=80, n_freq=40)
        
        # 计算幅度
        st_amplitude = np.abs(st_data)
        st_results.append(st_amplitude)
    
    return np.array(st_results)  # (epochs, channels, frequencies, samples)

# 询问用户输入时间窗口大小
try:
    user_epoch_duration = float(input("请输入常规时间窗口大小（秒）[默认60.0秒]: ") or epoch_duration)
    epoch_duration = user_epoch_duration if user_epoch_duration > 0 else epoch_duration
    print(f"使用常规时间窗口：{epoch_duration}秒")
    
    # 用户输入发作前后时间
    user_pre_seizure_time = float(input(f"请输入发作前时间窗口（秒）[默认{pre_seizure_time}秒]: ") or pre_seizure_time)
    pre_seizure_time = user_pre_seizure_time if user_pre_seizure_time > 0 else pre_seizure_time
    
    user_post_seizure_time = float(input(f"请输入发作后时间窗口（秒）[默认{post_seizure_time}秒]: ") or post_seizure_time)
    post_seizure_time = user_post_seizure_time if user_post_seizure_time > 0 else post_seizure_time
    
    print(f"发作前窗口：{pre_seizure_time}秒，发作后窗口：{post_seizure_time}秒")
except ValueError:
    print(f"输入无效，使用默认参数")

# 询问用户是否要考虑注释
consider_annotations = input("是否考虑癫痫发作注释？(y/n)[默认y]: ").lower() != 'n'
if not consider_annotations:
    print("将忽略所有注释信息，使用常规时间窗口划分")

# 遍历文件夹中的processed.fif文件
for file in os.listdir(folder_path):
    if file.endswith("_processed.fif"):
        file_path = os.path.join(folder_path, file)
        try:
            # 1. 读取原始数据
            raw = mne.io.read_raw_fif(file_path, preload=True)
            print(f"\n===== 处理文件: {file} =====")
            
            # 检查是否有癫痫发作注释
            seizures = []
            if consider_annotations:
                seizures = get_seizure_annotations(raw)
                if seizures:
                    print(f"发现 {len(seizures)} 个癫痫发作注释")
                else:
                    print("未发现癫痫发作注释 - 将使用常规分段")
            else:
                print("根据用户选择，忽略所有注释信息 - 使用常规分段")

            # 2. 低通滤波
            raw_filtered = raw.copy().filter(l_freq=None, h_freq=lowpass_freq, method="fir")

            # 3. 下采样
            raw_downsampled = raw_filtered.copy().resample(
                sfreq=target_sfreq, method="polyphase", n_jobs=-1, verbose=False
            )
            print(f"完成下采样至 {target_sfreq} Hz")

            # 4. 使用卡尔曼滤波进行基线校正
            data = raw_downsampled.get_data()
            num_channels = data.shape[0]
            processed_channels = Parallel(n_jobs=-1)(
                delayed(apply_kalman_filter)(data[i, :]) for i in range(num_channels)
            )
            data = np.array(processed_channels)
            raw_downsampled._data = data
            print("完成卡尔曼滤波基线校正")

            # 5. 使用Picard算法进行ICA（在连续数据上）
            try:
                n_components = 0.99  # 保留99%的方差
                ica = ICA(
                    n_components=n_components,
                    method='picard',
                    random_state=42,
                    max_iter=500
                )
                ica.fit(raw_downsampled)
                print("完成ICA拟合")

                # 分析ICA成分，识别更多伪迹成分
                components = ica.get_sources(raw_downsampled).get_data()
                peak_to_peak_values = [compute_peak_to_peak(comp) for comp in components]
                sfreq = raw_downsampled.info['sfreq']
                for i, component in enumerate(components):
                    freqs, psd = compute_psd(component, sfreq)
                    # 眼电伪迹判断
                    eog_like = np.sum(psd[(freqs >= 0) & (freqs < 5)]) / np.sum(psd) > 0.5
                    # 肌电伪迹判断
                    emg_like = np.sum(psd[(freqs >= 50)]) / np.sum(psd) > 0.3
                    # 心电伪迹判断
                    ecg_like = is_ecg_like(psd, freqs)
                    if eog_like or emg_like or ecg_like or peak_to_peak_values[i] > np.percentile(peak_to_peak_values, 95):
                        ica.exclude.append(i)

                print(f"检测到 {len(ica.exclude)} 个伪迹相关ICA成分")

                # 应用ICA校正到连续数据
                raw_cleaned = ica.apply(raw_downsampled.copy())
                print("完成ICA伪迹移除")
            except Exception as e:
                print(f"ICA处理失败，使用未经ICA处理的数据继续: {e}")
                raw_cleaned = raw_downsampled.copy()  # 使用未经ICA处理的数据继续

            # 对连续数据进行Z-分数归一化（按通道）
            raw_normalized = apply_zscore(raw_cleaned)
            print("完成Z-分数归一化")

            # 保存连续数据结果
            output_file = file_path.replace("_processed.fif", f"_processed_full_{target_sfreq}Hz.fif")
            raw_normalized.save(output_file, overwrite=True)
            print(f"保存连续数据至: {output_file}，采样率: {target_sfreq} Hz")
            
            # 创建考虑癫痫发作的epochs
            epochs_all_types = {}
            if consider_annotations and seizures:
                print("使用癫痫发作信息创建特殊分段...")
                main_epochs, epochs_dict = create_epochs_with_seizure_info(raw_normalized, epoch_duration, seizures, pre_seizure_time, post_seizure_time)
                if epochs_dict:
                    epochs_all_types = epochs_dict
                    print(f"成功创建不同类型的epochs")
                    
                    # 保存各种类型的epochs到单独的文件
                    for epoch_type, epoch_obj in epochs_dict.items():
                        if epoch_type != 'seizure':  # 常规处理非发作epochs
                            type_output_file = file_path.replace("_processed.fif", f"_epoched_{epoch_type}_{int(epoch_duration)}s_{target_sfreq}Hz-epo.fif")
                            epoch_obj.save(type_output_file, overwrite=True)
                            print(f"保存{epoch_type} epochs数据至: {type_output_file}")
                            
                            # 对这种类型应用Stockwell变换
                            print(f"对{epoch_type} epochs应用Stockwell变换...")
                            st_results = apply_stockwell_to_epoch(epoch_obj, target_sfreq)
                            print(f"完成{epoch_type} Stockwell变换，结果形状: {st_results.shape}")
                            
                            # 归一化
                            st_mean = np.mean(st_results, axis=(2, 3), keepdims=True)
                            st_std = np.std(st_results, axis=(2, 3), keepdims=True)
                            st_std[st_std == 0] = 1.0
                            st_normalized = (st_results - st_mean) / st_std
                            
                            # 保存
                            st_output_file = file_path.replace("_processed.fif", f"_stockwell_{epoch_type}_{int(epoch_duration)}s_{target_sfreq}Hz.npy")
                            np.save(st_output_file, st_normalized)
                            print(f"保存{epoch_type} Stockwell变换结果至: {st_output_file}")
                            
                            # 保存标签
                            labels = np.ones(len(epoch_obj)) * (0 if epoch_type == 'regular' else 
                                              1 if epoch_type == 'pre-seizure' else 
                                              3 if epoch_type == 'post-seizure' else 2)
                            labels_file = file_path.replace("_processed.fif", f"_labels_{epoch_type}_{int(epoch_duration)}s.npy")
                            np.save(labels_file, labels)
                        else:  # 特殊处理发作epochs列表
                            for i, seizure_epoch in enumerate(epoch_obj):
                                seizure_output_file = file_path.replace("_processed.fif", f"_epoched_seizure_{i+1}_{target_sfreq}Hz-epo.fif")
                                seizure_epoch.save(seizure_output_file, overwrite=True)
                                print(f"保存发作{i+1} epochs数据至: {seizure_output_file}")
                                
                                # 对发作应用Stockwell变换
                                print(f"对发作{i+1} epochs应用Stockwell变换...")
                                st_results = apply_stockwell_to_epoch(seizure_epoch, target_sfreq)
                                print(f"完成发作{i+1} Stockwell变换，结果形状: {st_results.shape}")
                                
                                # 归一化
                                st_mean = np.mean(st_results, axis=(2, 3), keepdims=True)
                                st_std = np.std(st_results, axis=(2, 3), keepdims=True)
                                st_std[st_std == 0] = 1.0
                                st_normalized = (st_results - st_mean) / st_std
                                
                                # 保存
                                st_output_file = file_path.replace("_processed.fif", f"_stockwell_seizure_{i+1}_{target_sfreq}Hz.npy")
                                np.save(st_output_file, st_normalized)
                                print(f"保存发作{i+1} Stockwell变换结果至: {st_output_file}")
                                
                                # 保存标签
                                labels = np.ones(len(seizure_epoch)) * 2  # 标签2表示发作
                                labels_file = file_path.replace("_processed.fif", f"_labels_seizure_{i+1}.npy")
                                np.save(labels_file, labels)
                    
                    # 保存一个所有非发作epochs合并的版本(可选)
                    non_seizure_epochs = []
                    for epoch_type, epoch_obj in epochs_dict.items():
                        if epoch_type != 'seizure' and len(epoch_obj) > 0:
                            non_seizure_epochs.append(epoch_obj)
                    
                    if non_seizure_epochs:
                        try:
                            combined_epochs = mne.concatenate_epochs(non_seizure_epochs)
                            combined_output_file = file_path.replace("_processed.fif", f"_epoched_non_seizure_{target_sfreq}Hz-epo.fif")
                            combined_epochs.save(combined_output_file, overwrite=True)
                            print(f"保存合并的非发作epochs至: {combined_output_file}")
                            
                            # 保存非发作标签
                            event_labels = combined_epochs.events[:, 2]
                            combined_labels_file = file_path.replace("_processed.fif", f"_labels_non_seizure.npy")
                            np.save(combined_labels_file, event_labels)
                        except Exception as e:
                            print(f"合并非发作epochs失败: {e}")
                else:
                    # 如果没有成功创建分类的epochs，使用main_epochs继续
                    epochs = main_epochs
                    print(f"创建了 {len(epochs)} 个epochs")
                    
                    # 保存epochs数据
                    epochs_output_file = file_path.replace("_processed.fif", f"_epoched_{int(epoch_duration)}s_{target_sfreq}Hz-epo.fif")
                    epochs.save(epochs_output_file, overwrite=True)
                    print(f"保存epoched数据至: {epochs_output_file}")
                    
                    # 应用Stockwell变换
                    print("开始对每个epoch计算Stockwell变换...")
                    st_results = apply_stockwell_to_epoch(epochs, target_sfreq)
                    print(f"完成Stockwell变换，结果形状: {st_results.shape}")
                    
                    # 对Stockwell变换结果进行归一化
                    st_mean = np.mean(st_results, axis=(2, 3), keepdims=True)
                    st_std = np.std(st_results, axis=(2, 3), keepdims=True)
                    st_std[st_std == 0] = 1.0
                    st_normalized = (st_results - st_mean) / st_std
                    
                    # 保存Stockwell变换结果
                    st_output_file = file_path.replace("_processed.fif", f"_stockwell_{int(epoch_duration)}s_{target_sfreq}Hz.npy")
                    np.save(st_output_file, st_normalized)
                    print(f"保存Stockwell变换结果至: {st_output_file}")
            else:
                # 使用常规分段
                print("\n对该文件使用常规时间窗口分段处理...")
                epochs = create_nonoverlapping_epochs(raw_normalized, epoch_duration)
                print(f"创建了 {len(epochs)} 个常规不重叠epochs，每个时长 {epoch_duration} 秒")
                
                # 保存epochs数据
                epochs_output_file = file_path.replace("_processed.fif", f"_epoched_regular_{int(epoch_duration)}s_{target_sfreq}Hz-epo.fif")
                epochs.save(epochs_output_file, overwrite=True)
                print(f"保存epoched数据至: {epochs_output_file}")
                
                # 应用Stockwell变换
                print("开始对每个epoch计算Stockwell变换...")
                st_results = apply_stockwell_to_epoch(epochs, target_sfreq)
                print(f"完成Stockwell变换，结果形状: {st_results.shape}")
                
                # 对Stockwell变换结果进行归一化
                st_mean = np.mean(st_results, axis=(2, 3), keepdims=True)
                st_std = np.std(st_results, axis=(2, 3), keepdims=True)
                st_std[st_std == 0] = 1.0
                st_normalized = (st_results - st_mean) / st_std
                
                # 保存Stockwell变换结果
                st_output_file = file_path.replace("_processed.fif", f"_stockwell_regular_{int(epoch_duration)}s_{target_sfreq}Hz.npy")
                np.save(st_output_file, st_normalized)
                print(f"保存Stockwell变换结果至: {st_output_file}")
                
                # 保存标签 (全部为0表示常规)
                labels = np.zeros(len(epochs))
                labels_file = file_path.replace("_processed.fif", f"_labels_regular_{int(epoch_duration)}s.npy")
                np.save(labels_file, labels)
                print(f"保存标签数据至: {labels_file}")
            
            # 生成示例可视化（可选）
            visualize_results = False  # 设置为True可开启可视化
            if visualize_results and epochs_all_types:
                # 为每种类型生成一个可视化示例
                for epoch_type, epoch_obj in epochs_all_types.items():
                    if epoch_type != 'seizure':
                        if len(epoch_obj) > 0:
                            # 获取第一个epoch的Stockwell变换
                            st_data = stockwell_transform(epoch_obj.get_data()[0], fs=target_sfreq)
                            st_amplitude = np.abs(st_data)
                            
                            plt.figure(figsize=(12, 8))
                            plt.imshow(
                                st_amplitude[0],  # 第一个通道
                                aspect='auto', 
                                origin='lower',
                                extent=[0, epoch_obj.times[-1], 0, 80],
                                cmap='viridis'
                            )
                            plt.colorbar(label='幅度')
                            plt.xlabel('时间 (秒)')
                            plt.ylabel('频率 (Hz)')
                            plt.title(f'Stockwell变换 - 文件: {file}, 通道: {epoch_obj.ch_names[0]}, 类型: {epoch_type}')
                            
                            viz_file = file_path.replace("_processed.fif", f"_stockwell_viz_{epoch_type}_{target_sfreq}Hz.png")
                            plt.savefig(viz_file, dpi=300)
                            plt.close()
                            print(f"保存{epoch_type}可视化结果至: {viz_file}")
                    else:  # 处理发作列表
                        for i, seizure_epoch in enumerate(epoch_obj):
                            if len(seizure_epoch) > 0:
                                st_data = stockwell_transform(seizure_epoch.get_data()[0], fs=target_sfreq)
                                st_amplitude = np.abs(st_data)
                                
                                plt.figure(figsize=(12, 8))
                                plt.imshow(
                                    st_amplitude[0],
                                    aspect='auto', 
                                    origin='lower',
                                    extent=[0, seizure_epoch.times[-1], 0, 80],
                                    cmap='viridis'
                                )
                                plt.colorbar(label='幅度')
                                plt.xlabel('时间 (秒)')
                                plt.ylabel('频率 (Hz)')
                                plt.title(f'Stockwell变换 - 文件: {file}, 通道: {seizure_epoch.ch_names[0]}, 类型: 发作{i+1}')
                                
                                viz_file = file_path.replace("_processed.fif", f"_stockwell_viz_seizure{i+1}_{target_sfreq}Hz.png")
                                plt.savefig(viz_file, dpi=300)
                                plt.close()
                                print(f"保存发作{i+1}可视化结果至: {viz_file}")
            elif visualize_results and len(epochs) > 0:
                plt.figure(figsize=(12, 8))
                plt.imshow(
                    st_results[0, 0], 
                    aspect='auto', 
                    origin='lower',
                    extent=[0, epoch_duration, 0, 80],
                    cmap='viridis'
                )
                plt.colorbar(label='幅度')
                plt.xlabel('时间 (秒)')
                plt.ylabel('频率 (Hz)')
                plt.title(f'Stockwell变换 - 文件: {file}, 通道: {epochs.ch_names[0]}')
                
                viz_file = file_path.replace("_processed.fif", f"_stockwell_viz_regular_{target_sfreq}Hz.png")
                plt.savefig(viz_file, dpi=300)
                plt.close()
                print(f"保存可视化结果至: {viz_file}")

            print(f"\n===== 文件 {file} 处理完成 =====\n")

        except Exception as e:
            print(f"处理失败: {file}，错误信息: {e}")
            import traceback
            traceback.print_exc()  # 打印完整错误堆栈
            continue

print("\n所有文件处理完成!")